<a href="https://colab.research.google.com/github/Sheriff414/my_deeplearning_model/blob/master/tiny_imagenet_proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tinyimagenet
import logging
logging.basicConfig(level=logging.INFO)
from pathlib import Path
from tinyimagenet import TinyImageNet

In [ ]:
import matplotlib.pyplot as plt

split ="val" # choose from "train", "val", "test"
dataset_path="~/.torchvision/tinyimagenet/"
dataset = TinyImageNet(Path(dataset_path),split=split)
n = len(dataset)
print(f"TinyImageNet, split {split}, has  {n} samples.")

100%|██████████| 248100043/248100043 [00:23<00:00, 10517369.46it/s]


Extracting /root/.torchvision/tinyimagenet/tiny-imagenet-200.zip to /root/.torchvision/tinyimagenet


TinyImageNet, split val, has  10000 samples.


In [ ]:
from torchvision import transforms as T

normalize_transform = T.Compose([ T.ToTensor(),
                                 T.Normalize(mean=TinyImageNet.mean,
                            std=TinyImageNet.std),
                             # Converting cropped images to tensors
])
train_transform = T.Compose([ T.Resize(256), # Resize images to 256 x 256
                T.CenterCrop(224), # Center crop image
                T.RandomHorizontalFlip(),
                normalize_transform

                ])


train = TinyImageNet(Path('./data'),split="train",transform=train_transform,imagenet_idx=True)
val = TinyImageNet(Path('./data'),split="val",transform=normalize_transform,imagenet_idx=True)
test = TinyImageNet(Path('./data'),split="test",transform=normalize_transform,imagenet_idx=True)
print(f'Dataset has {len(train.classes)} classes. Sample classes: {train.classes[:5]}')


100%|██████████| 248100043/248100043 [00:13<00:00, 18733632.31it/s]


Extracting data/tiny-imagenet-200.zip to data


Dataset has 200 classes. Sample classes: ['n01443537', 'n01629819', 'n01641577', 'n01644900', 'n01698640']


In [ ]:
train_batch_size=128
batch_size = 32
from torch.utils.data import DataLoader
train_loader = DataLoader(train,batch_size=train_batch_size,shuffle=True,pin_memory=True)
val_loader = DataLoader(val,batch_size=batch_size,shuffle=False,pin_memory=True)
test_loader = DataLoader(test,batch_size=batch_size,shuffle=False,pin_memory=True)

In [ ]:
train_loader

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transformsn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#Teacher model RESNET34
bmodel=torchvision.models.resnet34(pretrained=True)
num_ftrs=bmodel.fc.in_features
bmodel.fc=nn.Linear(num_ftrs, 200)
bmodel = bmodel.to(device)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100%|██████████| 83.3M/83.3M [00:01<00:00, 85.4MB/s]


In [ ]:
import torch.optim as optim
from torch.optim import lr_scheduler
#from einops.layers.torch import Reduce
import os
from torch.utils.data import DataLoader

# Hyper-parameters
num_epochs = 100
learning_rate = 0.01
criterionA = nn.CrossEntropyLoss()
optimizerA = torch.optim.SGD(bmodel.parameters(), momentum= 0.9, weight_decay=0.0005, lr=learning_rate) #weight_decay=0.005
n_total_steps = len(train_loader)
scheduler = lr_scheduler.MultiStepLR(optimizerA, milestones=(30, 70), gamma=0.1)
#scheduler = lr_scheduler.StepLR(optimizerA, step_size=2, gamma=0.1)


#TRAINING BASE MODEL



for epoch in range(num_epochs): #I decided to train the model for 50 epochs
    # Decay Learning Rate
    scheduler.step()
    loss_var = 0

    for name,dataloader in [ ('train',train_loader),('val',val_loader)]:

      for (images, labels) in train_loader:
        images = images.to(device=device)
        labels = labels.to(device=device)
          ## Forward Pass
        optimizerA.zero_grad()
        scores = bmodel(images)
        loss = criterionA(scores,labels)
        loss.backward()
        optimizerA.step()
        loss_var += loss.item()
        if idx%64==0:
            print(f'Epoch [{epoch+1}/{num_epochs}] || Step [{idx+1}/{len(train_loader)}] || Loss:{loss_var/len(train_loader)}')
    print(f"Loss at epoch {epoch+1} || {loss_var/len(train_loader)}")

    with torch.no_grad():
        correct = 0
        samples = 0
        for idx, (images, labels) in enumerate(val_loader):
            images = images.to(device=device)
            labels = labels.to(device=device)
            outputs = bmodel(images)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum()
            samples += preds.size(0)
            #acc= correct/samples*100
        print(f"accuracy {float(correct) / float(samples) * 100:.2f} percentage || Correct {correct} out of {samples} samples")
    #schedulerA.step(acc)
    PATH1 = '/tinyimagenet-resnet34.pt'
    torch.save(bmodel.state_dict(), PATH1)

print('Finished Training')


/usr/local/lib/python3.10/dist-packages/torch/optim/lr_scheduler.py:136: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


RuntimeError: ignored

In [ ]:
BATCH_SIZE=500
VAL_BATCH_SIZE=500
image_train=read_train_data()
image_val=read_validate_data()
LR=0.01


optimizer = torch.optim.Adam(bmodel.parameters(), lr=LR)   # optimize all cnn parameters
loss_func = nn.CrossEntropyLoss()

for epoch in range(10):
    step=0
    adjust_learning_rate(optimizer, epoch)
    for batch in train_batch_load(batch_size=BATCH_SIZE):
	#使用cuda
        b_x = Variable(batch[0].cuda())
        b_y = Variable(batch[1].cuda())
        output = bmodel(b_x)
        loss = loss_func(output, b_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if(step%50==0):
            for batch_val in val_batch_load(batch_size=VAL_BATCH_SIZE):
		#使用cuda，关闭BP
                x_test=Variable(batch_val[0].cuda(), volatile=True)
                y_test = Variable(batch_val[1].cuda())
                test_output = bmodel(x_test)
                pred = test_output.data.max(1, keepdim=True)[1]
                correct = pred.eq(y_test.data.view_as(pred)).cpu().sum()/VAL_BATCH_SIZE
                print('Epoch: ', epoch, 'step:', step, '| test accuracy: %.2f' % correct)
                break
        step = step + 1